In [ ]:
import os

print("CWD:", os.getcwd())
print("scripts exists:", os.path.exists("./scripts"))
print("csv exists:", os.path.exists("./winequality-red.csv"))


In [ ]:
# Installs only if SageMaker import is broken in this kernel image.
import sys
import subprocess

try:
    import boto3
    import sagemaker
    from sagemaker.sklearn.estimator import SKLearn
except Exception as e:
    print("Initial import failed, upgrading SageMaker SDK:", e)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "sagemaker"])
    import boto3
    import sagemaker
    from sagemaker.sklearn.estimator import SKLearn

print("sagemaker version:", sagemaker.__version__)


In [ ]:
session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()

# Use explicit role for this notebook instance.
role = "arn:aws:iam::047557961235:role/service-role/AmazonSageMaker-ExecutionRole-20260221T114423"

print("region:", region)
print("bucket:", bucket)
print("role:", role)


In [ ]:
local_csv = "winequality-red.csv"
assert os.path.exists(local_csv), f"CSV not found: {local_csv}"

s3_prefix = "ana680/wine-quality/data"
train_s3_uri = session.upload_data(path=local_csv, key_prefix=s3_prefix)
print("train_s3_uri:", train_s3_uri)


In [ ]:
# Part A: built-in SKLearn estimator (no custom container).
sk_estimator = SKLearn(
    entry_point="train_2a.py",
    source_dir="scripts",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session,
)

print("Estimator ready")


In [ ]:
sk_estimator.fit({"train": train_s3_uri})
print("Training complete")


In [ ]:
predictor_2a = sk_estimator.deploy(initial_instance_count=1, instance_type="ml.m5.large")
print("Endpoint:", predictor_2a.endpoint_name)


In [ ]:
# Sample values for the 11 input columns, important they are in same order
sample = [[7.4, 0.70, 0.00, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]]
pred = predictor_2a.predict(sample)
print("Prediction:", pred)


In [ ]:
# Apparently this can be run after done running everything to stop the charges
# Still need to understand how $ charges even work in this platform... 
# I never gave a cc or anything yet either? Where do we even go view charges? 
# Investigate further later once this initial training .ipynb notebook is working
# Oh! The video from 2/19 I think will be a tremendous resource now as well 
# Reviewed the other 2 linked in the content already, but his actual lecture video is found here
# https://navigate.nu.edu/d2l/lms/news/main.d2l?ou=59967
predictor_2a.delete_endpoint()
print("Endpoint deleted")
